# 교안 02. 자동 생성한 그래프의 근거와 품질을 검증합니다

**교안 01에서는 pandas 문서와 의료 논문을 각각 그래프로 만들고, 평가용 결과를 저장한 뒤 같은 이름의 노드를 통합했습니다.**  
1~3절은 pandas 예시로 앞에서 배운 품질 검사를 복습합니다.  
4절에서는 pandas의 청크 설정을 바꿔 다시 실행하고 같은 골드로 비교합니다.  
마지막 핵심 코드는 이 과정을 의료 논문에 적용한 완성 코드입니다.  

- **입력:** 본문은 `baseline_demo.json`과 `demo_gold.json`, 마지막 핵심 코드는 `baseline_followalong.json`과 `followalong_gold.json`을 사용합니다.
- **할 일:** 저장 단계와 품질 검사 복습 -> 청크 크기와 겹침을 늘려 재실행 -> 같은 골드로 비교.
- **결과:** 각 실행의 점수와, 잘못 뽑거나 놓친 관계 목록.


**실습의 목표**  

**1. 무엇을 평가하는지 정합니다**  

- LLM이 처음 뽑은 관계와, 허용 규칙에 맞춰 가지치기한 뒤 저장한 관계를 구분할 수 있습니다.

**2. 저장 관계의 스키마와 근거를 확인합니다**  

- (2-1) 저장 관계 전체에서 두 검사를 각각 계산할 수 있습니다.
- (2-2) 근거 문자열의 일치와 관계 의미의 적합성을 구분할 수 있습니다.

**3. 같은 문서의 골드와 비교합니다**  

- 고유 관계의 TP, FP, FN으로 정밀도, 재현율, F1을 계산할 수 있습니다.

**4. 청크 크기와 겹침을 늘려 다시 검증합니다**  

- (4-1) 원문, 스키마, 모델과 골드는 유지하고 청크 크기와 겹침을 늘립니다.
- (4-2) 점수와 실제 오류를 함께 읽어 변경의 효과를 설명할 수 있습니다.

마지막 부록에서는 `LLMGraphTransformer`의 입력과 반환값을 짧게 비교합니다.  

#### 사용할 라이브러리 불러오기

본문과 부록의 import를 먼저 실행합니다. 모델 생성과 DB 연결은 4절에서 합니다.

In [ ]:
import json
from pathlib import Path
from pprint import pprint
from copy import deepcopy
from neo4j_graphrag.experimental.components.schema import GraphSchema
from neo4j_graphrag.experimental.components.types import (
    Neo4jGraph,
    Neo4jNode,
    Neo4jRelationship,
)
from neo4j_graphrag.experimental.components.graph_pruning import GraphPruning
import os
from urllib.parse import urlsplit
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from functools import partial
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from neo4j_graphrag.experimental.components.text_splitters.langchain import (
    LangChainTextSplitterAdapter,
)
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from uuid import uuid4
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_neo4j import LLMGraphTransformer

#### 자료 경로 준비

pandas 저장본과 골드의 파일 경로를 준비합니다. DB와 API 연결은 4절에서 합니다.  

In [ ]:
# 학생용은 현재 폴더, 정답은 한 단계 위 폴더의 자료를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(path):
    """JSON 파일 하나를 파이썬 사전 또는 목록으로 읽습니다."""
    # path는 파일 위치이며, UTF-8로 읽어 한글을 유지합니다.
    return json.loads(path.read_text(encoding="utf-8"))


def save_json(path, value):
    """실행 결과를 한글을 유지한 JSON 파일로 저장합니다."""
    # value는 저장할 사전이나 목록입니다. ensure_ascii=False는 한글을 문자 그대로 남깁니다.
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

## 1. 무엇을 평가하는지 정합니다

**교안 01의 파일에는 스키마 검사를 거친 뒤, 중복 노드를 합치기 전에 기록한 관계가 들어 있습니다.**  
교안 01은 스키마로 추출을 안내하고, `GraphPruning`으로 허용 타입과 관계의 양 끝 조합을 검사합니다.  

**가지치기(pruning)** 는 스키마에서 허용하지 않은 노드, 관계나 속성을 제외하는 작업입니다.  
**이 교안의 평가 대상은 가지치기를 통과해 DB에 저장된 관계입니다.** 제외된 관계는 저장본에 들어 있지 않습니다.  

**예:** LLM이 10건을 추출하고 가지치기로 2건을 제외했다면, 여기서는 저장된 8건을 평가합니다.  

### 불러올 저장본의 구조를 확인합니다

`baseline_demo.json`에서 다음 세 항목을 읽습니다.  
`document`는 원문과 출처, `rows`는 평가할 관계, `chunks`는 당시 나눈 원문 청크입니다.  

#### pandas 2.0.3 실행 범위 확인

원문과 관계는 같은 실행 저장본에서 함께 읽습니다.  

In [ ]:
# baseline_demo.json: 교안 01에서 실제 실행한 설정, 원문, DB 조회 결과입니다.
demo_snapshot = read_json(output_dir / "baseline_demo.json")
demo_doc = demo_snapshot["document"]
demo_rows = demo_snapshot["rows"]
print("평가 단계:", demo_snapshot["stage"])
print("원문:", demo_doc["doc_id"], "/ 저장 관계:", len(demo_rows))

#### 가지치기가 하는 일 확인

가지치기 동작을 보려고 정상 관계와 `ApiElement -> FIXES_API -> ApiElement` 오류를 직접 만듭니다. 둘 다 타입 이름은 허용되지만 두 번째 관계는 양 끝 조합이 틀렸습니다.  

- `Neo4jGraph`: 노드와 관계 목록을 담는 Python 객체입니다. 아직 DB에 저장한 상태가 아닙니다.
- `GraphSchema.model_validate(schema)`: 스키마 사전의 형식을 검사하고 빌더가 사용할 객체로 바꿉니다.
- `GraphPruning.run(...)`: 그 규칙에 맞게 가지치기한 그래프와 제외 사유를 반환합니다.

In [ ]:
# 저장본에서 당시 추출에 사용한 스키마를 읽어 같은 규칙으로 가지치기합니다.
schema = demo_snapshot["schema"]

# (1) 가지치기 동작을 관찰하려고 정상 관계와 양 끝 타입이 틀린 관계를 직접 만듭니다.
# 노드 id는 아래 start_node_id와 end_node_id가 가리키는 양 끝입니다.
pruning_input = Neo4jGraph(
    nodes=[
        Neo4jNode(id="release", label="Release", properties={"name": "2.0.3"}),
        Neo4jNode(
            id="api", label="ApiElement", properties={"name": "DataFrame.to_string"}
        ),
    ],
    relationships=[
        # 정상: Release에서 ApiElement로 향하는 FIXES_API 관계입니다.
        Neo4jRelationship(
            start_node_id="release",
            end_node_id="api",
            type="FIXES_API",
            properties={"evidence": demo_doc["paragraphs"][0]["text"]},
        ),
        # 오류: 주어가 Release여야 하는데 ApiElement(id="api")를 사용했습니다.
        Neo4jRelationship(
            start_node_id="api",
            end_node_id="api",
            type="FIXES_API",
            properties={"evidence": demo_doc["paragraphs"][0]["text"]},
        ),
    ],
)
# (2) graph는 검사할 그래프, schema는 적용할 규칙입니다. model_validate는 사전을 검사 객체로 바꿉니다.
# 규칙을 검사 객체로 바꾸는 동안 저장용 schema가 바뀌지 않도록 복사본을 전달합니다.
pruned = await GraphPruning().run(
    graph=pruning_input, schema=GraphSchema.model_validate(deepcopy(schema))
)
# (3) graph에는 남은 결과, pruning_stats에는 제외한 항목과 사유가 있습니다.
print(
    "가지치기 전 관계:",
    len(pruning_input.relationships),
    "/ 가지치기 후:",
    len(pruned.graph.relationships),
)
for item in pruned.pruning_stats.pruned_relationships:
    print("제거한 관계:", item.item.type, "/ 사유:", item.pruned_reason.value)

가지치기 후 남은 관계가 하나라는 것은 **허용한 구조만 남았다는 뜻**입니다. 관계의 의미까지 검증한 결과는 아닙니다.  
예를 들어 `ApiElement -> FIXES_API -> Release`를 허용한 방향으로 뒤집을 수도 있습니다.  
**방향을 고쳤다고 수정 사실까지 입증되는 것은 아니므로 원문을 확인해야 합니다.**  
[Graph pruner 공식 설명](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_kg_builder.html#schema-guidance-and-graph-filtering)  

## 2. 저장 관계의 스키마와 근거를 확인합니다

### 2-1. 두 검사를 같은 전체 행에 각각 적용합니다

| 지표 | 확인할 질문 | 분자 ÷ 분모 |
|---|---|---|
| 저장 관계의 스키마 준수율 | 허용한 관계 이름과 양 끝 타입을 지켰나요? | 허용한 타입과 관계 조합을 지킨 행 ÷ 저장 관계 전체 행 |
| 저장 관계의 근거 원문 일치율 | 인용문이 이 출처에 그대로 있나요? | 빈칸이 아닌 `evidence`가 출처 원문에 있는 행 ÷ 같은 전체 행 |

교안 01의 `SimpleKGPipeline`은 **`GraphPruning`으로 스키마를 자동 검사한 뒤 저장**합니다.  

- **저장 후 스키마 준수율:** 저장 관계가 있고 같은 규칙으로 검사하면 100%가 정상입니다. 저장 결과 확인용이며, LLM의 가지치기 전 추출 품질을 뜻하지 않습니다.
- **근거 원문 일치율과 골드 정밀도, 재현율:** 인용 오류, 잘못된 관계나 누락이 있으면 낮을 수 있습니다. 3절에서 골드와 비교합니다.

**근거 검사는 스키마 검사 결과와 관계없이 저장 관계 전체에 적용합니다. 평가 목록이나 골드를 줄이지 않습니다.**  

**인용 검사는 `document["text"]`에 있는 문서 전체 원문과 대조합니다.**  
`row["chunk_texts"]`는 주어와 목적어가 공통으로 연결된 청크 원문 목록이며, 근거 주변 문맥을 읽을 때 참고합니다.  

여기서 **한 행은 DB에 저장된 관계 한 건**입니다. 같은 트리플이라도 DB 관계 ID가 다르면 별도 행으로 검사합니다. 3절의 골드 비교에서는 중복 트리플을 한 번만 셉니다.  

#### 검사와 관계 키 함수 준비

- `schema_ok(row, patterns)`: 해당 자료의 허용 조합과 맞으면 `True`입니다.
- `evidence_ok(row, document)`: 출처 ID가 같고, 빈칸이 아닌 인용문이 원문에 그대로 있으면 `True`입니다.
- `triple_key(row)`: 비교할 세 값만 꺼냅니다. 예: `("2.0.3", "FIXES_API", "read_csv")`.

`sum`으로 `True`를 세면 검사를 통과한 행 수가 됩니다.  

In [ ]:
def schema_ok(row, patterns):
    """한 행의 타입과 관계가 patterns에 지정한 조합인지 검사합니다."""
    return (row["subject_type"], row["relation"], row["object_type"]) in [
        tuple(pattern) for pattern in patterns
    ]


def evidence_ok(row, document):
    """근거가 비어 있지 않고 같은 문서의 전체 원문에 그대로 있으면 True입니다.

    document는 원문 문서 사전입니다. 개별 청크는 검사하지 않으므로,
    다른 청크의 문장을 인용했는지나 관계의 의미가 맞는지는 판정하지 않습니다.
    """
    # (1) row는 저장 관계 한 행, document는 대조할 출처 문서입니다.
    if row["source_doc_id"] != document["doc_id"]:
        return False

    # (2) 문자열이 아니거나 공백뿐이면 인용문으로 인정하지 않습니다.
    evidence = row["evidence"]
    if not isinstance(evidence, str) or not evidence.strip():
        return False

    # (3) 전체 문서에서 인용 문자열을 찾습니다. 출처 청크 내부의 일치는 별도 검사입니다.
    return evidence in document["text"]


def triple_key(row):
    """정밀도와 재현율에서 한 관계로 세는 (주어, 관계, 목적어)를 반환합니다."""
    return row["subject"], row["relation"], row["object"]

#### pandas 2.0.3 저장 관계의 두 비율

저장 관계가 있는 실행에서 같은 전체 행 수를 분모로 두 비율을 계산합니다.  

In [ ]:
# 두 비율 모두 저장 관계 전체를 분모로 사용합니다. 검사 때문에 행을 삭제하지 않습니다.
# sum은 검사 결과 True를 1로 세어 통과 행 수를 구합니다.
demo_schema_count = sum(
    schema_ok(row, demo_snapshot["schema"]["patterns"]) for row in demo_rows
)
demo_evidence_count = sum(evidence_ok(row, demo_doc) for row in demo_rows)
demo_total = len(demo_rows)
print("스키마 준수 행:", demo_schema_count, "/", demo_total)
print("근거 원문 일치 행:", demo_evidence_count, "/", demo_total)
print(f"저장 관계의 스키마 준수율: {demo_schema_count / demo_total:.2%}")
print(f"저장 관계의 근거 원문 일치율: {demo_evidence_count / demo_total:.2%}")

### 2-2. 인용 검사와 의미 판정을 구분합니다

- **근거 원문 일치:** `evidence`의 문구가 출처 원문에 그대로 있나요?
- **관계의 의미:** 원문이 추출된 주어, 관계, 목적어를 뒷받침하나요?

| 예시 | 원문 일치 | 관계의 의미 |
|---|---|---|
| 부정한 문장을 그대로 인용하고, 사실인 것처럼 관계를 추출 | 통과 | 틀림 |
| 관계는 올바르게 추출했지만, 근거를 원문과 다른 표현으로 요약 | 불일치 | 맞을 수 있음 |

<br>

| 확인한 문제 | 교정 방향 |
|---|---|
| 근거가 비었거나 원문과 다름 | 관계를 확인한 뒤 원문 구절을 그대로 인용 |
| 주어 또는 목적어를 잘못 연결 | 주변 문맥에서 각 이름이 가리키는 대상을 확인 |
| 예정이나 부정을 사실로 추출 | 관계의 포함 기준과 제외 예시를 추출 지시에 반영 |

**인용이 원문과 같아도 관계가 맞다는 뜻은 아닙니다.** 근거와 주변 문맥을 함께 읽습니다.  

#### pandas 2.0.3 관계의 근거 읽기

저장한 모든 관계를 읽으면서 API 이름, 버전과 수정 내용을 대조합니다.  

In [ ]:
for row in demo_rows:
    print("트리플:", triple_key(row))
    print("근거:", row["evidence"])
    print("원문 일치:", evidence_ok(row, demo_doc))
    print("출처 청크:", row["chunk_texts"])
    print()

## 3. 같은 문서의 골드와 비교합니다

**골드는 원문과 포함 기준을 읽고 작성한 정답 트리플 목록입니다.**  

- **같은 범위:** 추출 결과와 골드는 같은 원문을 대상으로 합니다.
- **고유 관계:** 같은 주어, 관계, 목적어가 반복돼도 한 번만 셉니다.
- **완전일치:** 세 값의 문자열이 모두 같아야 맞힌 관계입니다. 표기를 바꾼 경우도 FP와 FN에 나타날 수 있습니다.

pandas 예시에서는 버그를 수정한 API만 정답에 포함하고 설치 옵션이나 단순 언급은 제외했습니다.  

| 구분 | 이번 비교의 뜻 |
|---|---|
| TP: 맞힘 | 저장 관계에도 골드에도 있는 관계 |
| FP: 잘못 뽑음 | 저장 관계에만 있는 관계 |
| FN: 놓침 | 골드에만 있는 관계 |

<br>

| 지표 | 확인할 질문 | 계산 |
|---|---|---|
| 정밀도 | 뽑은 것 중 맞는 비율은? | `TP / (TP + FP)` |
| 재현율 | 찾아야 할 정답 중 찾은 비율은? | `TP / (TP + FN)` |
| F1 | 두 비율을 함께 보면? | `2 × TP / (2 × TP + FP + FN)` |

**계산 예:** 정답 5관계 중 2개를 맞히고, 잘못된 관계 1개도 뽑았다면 정밀도는 `2/3`, 재현율은 `2/5`, F1은 `4/8 = 0.5`입니다.  
F1은 두 비율의 조화평균입니다. 한쪽만 높아서는 높은 점수를 얻기 어렵습니다.  
**저장 관계 전체를 비교하므로 앞의 검사 실패 행도 포함합니다.**  

#### pandas 2.0.3 골드와 검토 범위 확인

수정 항목 4개에서 API 5개를 정답으로 기록했습니다. `paragraph_id`로 근거 위치를 찾습니다.  

In [ ]:
# demo_gold.json: pandas 2.0.3의 원문 항목 4개를 읽고 작성한 정답 트리플 5건입니다.
demo_gold = read_json(data_dir / "demo_gold.json")
for row in demo_gold:
    print("원문 항목 ID:", row["paragraph_id"], "/ 정답 트리플:", triple_key(row))

# paragraphs는 정답을 만들 때 검토한 원문 항목 목록입니다.
# 원문 항목 하나에서 정답 트리플이 여러 건 나올 수 있으므로 두 수는 다를 수 있습니다.
print("검토한 원문 항목 수:", len(demo_doc["paragraphs"]))
print("정답으로 작성한 트리플 수:", len(demo_gold))

#### 고유 관계를 골드와 비교하는 함수

- **입력:** 저장 관계 전체와 같은 원문 범위의 골드입니다.
- **처리:** 같은 관계의 중복을 제거하고 TP, FP, FN을 셉니다.
- **반환:** 세 관계 집합과 정밀도, 재현율, F1입니다.

In [ ]:
def score_relations(rows, gold):
    """같은 문서 범위의 고유 관계를 골드와 비교합니다.

    Args:
        rows: 평가할 저장 관계 전체. 근거 검사 실패 행도 포함합니다.
        gold: 해당 입력 문서의 고정 정답 목록.
    Returns:
        TP, FP, FN 집합과 정밀도, 재현율, F1을 담은 사전.
    """
    # (1) 같은 관계가 여러 행에 있어도 한 번만 세도록 집합을 만듭니다.
    # 검사 실패 행도 predicted에 포함하고, 골드 목록은 줄이지 않습니다.
    predicted = {triple_key(row) for row in rows}
    expected = {triple_key(row) for row in gold}
    # (2) 맞힘, 잘못 뽑음, 놓침을 집합 비교로 나눕니다.
    tp = predicted & expected  # 양쪽에 있음: 맞힌 관계
    fp = predicted - expected  # 추출에만 있음: 잘못 뽑은 관계
    fn = expected - predicted  # 골드에만 있음: 놓친 관계

    # (3) 정밀도는 추출 수, 재현율은 정답 수로 나눕니다. 분모가 없으면 None입니다.
    precision = len(tp) / len(predicted) if predicted else None
    recall = len(tp) / len(expected) if expected else None
    # 같은 TP, FP, FN으로 F1도 계산합니다.
    denominator = 2 * len(tp) + len(fp) + len(fn)
    f1 = 2 * len(tp) / denominator if denominator else None
    # (4) 점수와 오류 관계를 함께 반환해, 낮은 점수의 원인도 확인할 수 있게 합니다.
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

#### pandas 2.0.3의 맞힘과 누락 계산

표기가 다르면 완전일치에서 다른 관계로 셉니다. FP와 FN을 읽어 표기 문제인지 의미 오류인지 구분하세요.  

In [ ]:
# 스키마나 근거 검사에서 실패한 저장 행도 비교합니다. 골드는 항상 같은 목록입니다.
demo_metrics = score_relations(demo_rows, demo_gold)
print(
    "TP / FP / FN:",
    len(demo_metrics["tp"]),
    "/",
    len(demo_metrics["fp"]),
    "/",
    len(demo_metrics["fn"]),
)
for metric in ["precision", "recall", "f1"]:
    value = demo_metrics[metric]
    print(metric + ":", "미산출" if value is None else f"{value:.4f}")
print("잘못 뽑은 관계:", sorted(demo_metrics["fp"]))
print("놓친 관계:", sorted(demo_metrics["fn"]))

## 4. 청크 크기와 겹침을 늘려 다시 검증합니다

### 4-1. 겹침 비율을 유지하며 청크를 키웁니다

관련 문장을 더 넓게 읽도록 **청크 크기와 겹침 문자 수를 함께 늘립니다.**  
겹침 비율은 `chunk_overlap / chunk_size`로 계산하며, **두 설정 모두 20%로 맞춥니다.**  

| 설정 | 기준 실행 | 변경 실행 |
|---|---:|---:|
| 청크 최대 크기 (`chunk_size`) | 500자 | 2,000자 |
| 겹침 목표 (`chunk_overlap`) | 100자 | 400자 |
| 겹침 비율 | 20% | 20% |

**원문, 스키마, 프롬프트, 모델과 골드는 그대로 유지합니다.**  
20%는 이번 실습의 설정 기준이며, 실제 겹침은 문단 경계에 따라 달라집니다.  


- **확인할 것:** 관련 문장이 함께 전달됐는지, 정밀도와 재현율이 어떻게 달라졌는지 비교합니다.
- **이번 원문:** pandas 원문은 2,000자보다 짧아 변경 후에는 청크가 하나입니다. 따라서 겹침 400자의 효과만 따로 확인하는 실험은 아닙니다.

[청크 크기와 겹침 공식 설명](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)  

#### 재실행 연결과 모델 준비

Neo4j에 연결하고, 교안 01과 같은 추출 모델과 임베딩 모델을 준비합니다.  

In [ ]:
# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()


def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 목록으로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]


# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print(
    "Neo4j 연결 완료. 호스트:",
    connection_address.hostname,
    "/ 포트:",
    connection_address.port,
)


llm = OpenAILLM(
    model_name="gpt-5.6-luna",  # 관계를 추출할 모델 이름입니다.
)
embedder = OpenAIEmbeddings(
    model="text-embedding-3-large",  # 벡터를 만들 모델 이름입니다.
)
# partial은 이후 embed_query를 호출할 때 dimensions=768을 항상 함께 전달합니다.
embedder.embed_query = partial(embedder.embed_query, dimensions=768)

#### 같은 추출 조건 복원

교안 01에서 실제 사용한 스키마와 프롬프트를 저장본에서 읽습니다. 분할 설정에 따른 차이를 비교하도록 추출 기준은 유지합니다.  

In [ ]:
schema = demo_snapshot["schema"]
prompt_template = demo_snapshot["prompt_template"]
print(
    "기준 크기 / 겹침:",
    demo_snapshot["chunk_size"],
    "/",
    demo_snapshot["chunk_overlap"],
)
print("변경할 크기 / 겹침: 2000 / 400")

#### pandas 2.0.3의 넓어진 청크 확인

앞 절에서 읽은 원문을 그대로 사용합니다.  

In [ ]:
demo_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,  # 청크 하나의 최대 문자 수입니다.
    chunk_overlap=400,  # 청크 크기의 20%를 겹침 목표로 설정합니다.
)
# adapter는 LangChain 분할 결과를 빌더가 받는 청크 묶음으로 바꿉니다.
demo_splitter = LangChainTextSplitterAdapter(demo_text_splitter)
demo_chunks = await demo_splitter.run(text=demo_doc["text"])
for chunk in demo_chunks.chunks:
    print("청크:", chunk.index, "/ 문자 수:", len(chunk.text))
    print(chunk.text)
    print()
print("청크 수:", len(demo_chunks.chunks))

#### pandas 2.0.3을 다시 실행

`execution_id`는 이번 실행을 구분하는 번호입니다. 새 번호로 저장하고, 조회할 때도 이 번호로 결과를 찾습니다.  

In [ ]:
demo_builder = SimpleKGPipeline(
    llm=llm,  # 원문에서 개체와 관계를 추출할 모델입니다.
    driver=driver,  # 생성한 그래프를 저장할 Neo4j 연결입니다.
    embedder=embedder,  # 청크의 검색용 벡터를 만드는 모델입니다.
    schema=deepcopy(schema),  # 허용 타입과 관계 규칙을 복사해 전달합니다.
    prompt_template=prompt_template,  # 추출 기준과 인용 규칙입니다.
    text_splitter=demo_splitter,  # 앞에서 확인한 청크 분할기입니다.
    from_file=False,  # 파일을 여는 대신 text로 받은 원문을 처리합니다.
    on_error="RAISE",  # 응답 처리에 실패하면 오류를 알리고 중단합니다.
    perform_entity_resolution=False,  # 같은 타입과 이름의 노드를 자동으로 합치지 않습니다.
)
print("준비한 청크 크기:", 2000)


# execution_id는 이번 실행의 결과만 조회하기 위한 번호이며 개체의 표준 ID가 아닙니다.
demo_execution_id = str(uuid4())
demo_result = await demo_builder.run_async(
    text=demo_doc["text"],  # 실제로 분할하고 추출할 원문입니다.
    file_path=demo_doc["url"],  # 문서 노드에 남길 출처 URL입니다. 접속하지 않습니다.
    # 원문 문서와 이번 실행을 구분할 정보를 문서 노드에 저장합니다.
    document_metadata={
        "source_doc_id": demo_doc["doc_id"],
        "execution_id": demo_execution_id,
    },
)
# ER을 끈 파이프라인의 마지막 단계는 writer입니다. 저장 실패를 완료로 처리하지 않습니다.
# writer는 그래프를 DB에 저장하는 구성요소이며 status는 저장 작업의 성공 여부입니다.
demo_writer_status = demo_result.result["writer"]["status"]
if demo_writer_status != "SUCCESS":
    raise RuntimeError(demo_result.result["writer"])
print("완료한 실행 ID:", demo_execution_id)
print("DB 저장 상태:", demo_writer_status)

#### 조회 함수 준비

원문 청크와 관계를 교안 01과 같은 방식으로 꺼냅니다.  

In [ ]:
def read_relations(execution_id):
    """지정한 실행에서 저장한 개체 간 관계를 평가용 사전 목록으로 읽습니다.

    Args:
        execution_id (str): 빌더를 실행할 때 Document에 저장한 실행 ID.
            demo_execution_id처럼, 조회하려는 실행에서 사용한 값을 전달합니다.

    Returns:
        list[dict]: 관계 ID별 트리플과 근거, 출처 정보. 결과가 없으면 [].
            같은 관계의 청크 원문은 chunk_texts 목록에 모읍니다.

    Example:
        반환 형태 예시입니다. 실제 DB 식별자는 다르며 원문은 설명을 위해 줄였습니다.
        [{
            "relationship_id": "관계 식별자 예시",
            "subject": "2.0.3",
            "subject_type": "Release",
            "relation": "FIXES_API",
            "object": "DataFrame.to_string",
            "object_type": "ApiElement",
            "evidence": "Fixed regression when DataFrame.to_string",
            "source_doc_id": "pandas_doc_source_whatsnew_v2_0_3",
            "chunk_texts": ["What's new in 2.0.3 ... Fixed regression when DataFrame.to_string ..."]
        }]
        rows[0]["object"]는 첫 관계의 목적어 이름이며,
        rows[0]["chunk_texts"][0]은 그 관계에 연결된 첫 번째 원문 문자열입니다.
    """
    return run_cypher(
        """
    // (1) 실행 ID로 문서 범위를 고르고, 그 문서의 청크와 주어 개체를 찾습니다.
    MATCH (d:Document {execution_id: $execution_id})<-[:FROM_DOCUMENT]-(c:Chunk)<-[:FROM_CHUNK]-(s:__Entity__)
    // (2) 같은 청크에 연결된 목적어를 찾습니다. 관계 타입은 제한하지 않습니다.
    MATCH (s)-[r]->(o:__Entity__)-[:FROM_CHUNK]->(c)
    // (3) AS 오른쪽 이름이 반환 사전의 키가 됩니다.
    RETURN
        elementId(r) AS relationship_id, // DB 안에서 관계를 구분하는 ID입니다.
        s.name AS subject, // 주어 노드의 이름입니다.
        head([x IN labels(s) WHERE NOT x STARTS WITH '__']) AS subject_type, // 관리 레이블을 제외한 첫 타입입니다.
        type(r) AS relation, // 주어에서 목적어로 향하는 관계 타입입니다.
        o.name AS object, // 목적어 노드의 이름입니다.
        head([x IN labels(o) WHERE NOT x STARTS WITH '__']) AS object_type, // 관리 레이블을 제외한 첫 타입입니다.
        coalesce(r.evidence, '') AS evidence, // 근거 인용문이며, 없으면 빈 문자열입니다.
        d.source_doc_id AS source_doc_id, // 원본 문서 ID입니다. 실행 ID와 다릅니다.
        collect(DISTINCT c.text) AS chunk_texts // 연결된 청크 원문을 중복 없이 모읍니다.
    ORDER BY subject, relation, object, relationship_id
    """,
        execution_id=execution_id,
    )


def read_chunks(execution_id):
    """지정한 실행에서 저장한 청크의 순서, 원문, 임베딩 차원 수를 읽습니다.

    Args:
        execution_id (str): 빌더를 실행할 때 Document에 저장한 실행 ID.
            문서 이름이나 source_doc_id가 아니라 demo_execution_id 같은 실행 값을 씁니다.

    Returns:
        list[dict]: 청크별 원문과 임베딩 차원 정보. 순번순으로 정렬하며, 없으면 [].

    Example:
        반환 형태 예시입니다. 원문은 설명을 위해 줄였으며 실제 청크 수와 내용은 다릅니다.
        [
            {"index": 0, "text": "What's new in 2.0.3 ...", "dimensions": 768},
            {"index": 1, "text": "Bug fixes ...", "dimensions": 768}
        ]
    """
    return run_cypher(
        """
    // (1) 지정한 실행의 문서에 연결된 청크만 고릅니다.
    MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document {execution_id: $execution_id})
    // (2) 청크 하나를 사전 하나로 읽습니다. AS 오른쪽이 사전의 키입니다.
    RETURN
        c.index AS index, // 문서 안의 청크 순번입니다. 0부터 시작합니다.
        c.text AS text, // 청크에 저장된 원문입니다.
        size(c.embedding) AS dimensions // 벡터 원소 수, 즉 임베딩 차원입니다.
    // 원문을 읽는 순서대로 확인할 수 있게 청크 순번으로 정렬합니다.
    ORDER BY index
    """,
        execution_id=execution_id,
    )

#### pandas 2.0.3 변경 결과 저장

`demo_before`는 기존 500자 실행의 저장본입니다. 새 2000자 결과는 `demo_snapshot`에 담아 둘을 비교합니다.  

기존 `baseline_demo.json`은 유지하고 새 결과는 `large_demo.json`으로 저장합니다.  

In [ ]:
demo_before = demo_snapshot
# 실행 ID가 같은 DB 관계와 청크를 각각 읽습니다. 아직 노드를 통합하기 전입니다.
demo_rows = read_relations(demo_execution_id)
demo_stored_chunks = read_chunks(demo_execution_id)
print("저장 관계 행 수:", len(demo_rows), "/ 청크 수:", len(demo_stored_chunks))
for chunk in demo_stored_chunks:
    print("청크:", chunk["index"], "/ 임베딩 차원:", chunk["dimensions"])
for row in demo_rows:
    print("관계:", row["subject"], "->", row["relation"], "->", row["object"])
    print("근거:", row["evidence"])
    print()

# 문서, 설정과 조회 결과를 함께 저장해 교안 02에서 같은 실행을 평가합니다.
demo_snapshot = {
    # 평가 대상이 중복 노드 통합 전 결과임을 기록합니다.
    "stage": "허용하지 않은 노드·관계·속성을 가지치기한 뒤 DB에 저장한 결과(중복 노드 통합 전)",
    # 어느 실행에서 만든 결과인지 구분합니다.
    "execution_id": demo_execution_id,
    # 전체 원문: 근거 인용을 검사하고, 같은 문서로 다시 추출할 때 사용합니다.
    "document": demo_doc,
    # 실행 설정: 비교 실험에서 바꿀 조건과 유지할 조건을 확인합니다.
    "chunk_size": 2000,
    "chunk_overlap": 400,
    "text_splitter": "RecursiveCharacterTextSplitter",
    "model": "gpt-5.6-luna",
    "embedding_model": "text-embedding-3-large",
    "dimensions": 768,
    "schema": schema,
    "prompt_template": prompt_template,
    "perform_entity_resolution": False,
    # 추출 관계: 스키마와 근거를 검사하고 골드와 비교합니다.
    "rows": demo_rows,
    # 당시 청크: 원문이 나뉜 위치와 변경 전후의 청크 수와 내용을 확인합니다.
    "chunks": demo_stored_chunks,
}
save_json(output_dir / "large_demo.json", demo_snapshot)
print("저장 파일:", output_dir / "large_demo.json")

### 4-2. 점수와 실제 오류를 함께 비교합니다

각 실행의 **저장 관계 전체**로 스키마와 근거 일치율을 계산합니다. 골드 비교에서는 그 목록의 고유 관계를 셉니다.  
정답을 더 많이 뽑았는지, 잘못 뽑은 관계도 늘었는지 함께 읽습니다.  

- **FN이 줄었다면:** 전에 놓친 어떤 정답 관계를 새로 찾았는지 확인합니다.
- **FP가 늘었다면:** 잘못된 관계를 뽑았는지, 이름 표기가 골드와 다른지 원문과 대조합니다.
- **근거 일치율만 올랐다면:** 인용은 개선됐지만 관계의 정확성도 좋아졌는지는 골드와 따로 비교합니다.

#### pandas 2.0.3 설정 전후 비교

같은 골드 5관계로 두 결과를 비교합니다. 큰 청크 결과를 정답으로 간주하지 않습니다.  

In [ ]:
for label, snapshot in [("500자", demo_before), ("2000자", demo_snapshot)]:
    rows = snapshot["rows"]
    document = snapshot["document"]
    print(
        "청크 설정:", label, "/ 저장 행:", len(rows), "/ 청크:", len(snapshot["chunks"])
    )
    print("크기 / 겹침:", snapshot["chunk_size"], "/", snapshot["chunk_overlap"])
    schema_count = sum(schema_ok(row, snapshot["schema"]["patterns"]) for row in rows)
    evidence_count = sum(evidence_ok(row, document) for row in rows)
    print("스키마 준수:", schema_count, "/", len(rows))
    print("근거 원문 일치:", evidence_count, "/", len(rows))
    if rows:
        print(
            f"스키마 준수율: {schema_count / len(rows):.2%} / 근거 원문 일치율: {evidence_count / len(rows):.2%}"
        )
    else:
        print("스키마 준수율: 미산출 / 근거 원문 일치율: 미산출")
    metrics = score_relations(rows, demo_gold)
    print(
        "TP / FP / FN:",
        len(metrics["tp"]),
        "/",
        len(metrics["fp"]),
        "/",
        len(metrics["fn"]),
    )
    for metric in ["precision", "recall", "f1"]:
        value = metrics[metric]
        print(metric + ":", "미산출" if value is None else f"{value:.4f}")
    print("FP:", sorted(metrics["fp"]))
    print("FN:", sorted(metrics["fn"]))
    print()

다른 문서에서도 효과가 있는지 확인하려면 설정을 조정할 때 보지 않은 문서와 그 골드로 평가합니다.  

#### 본문의 연결 종료

본문 비교를 마칩니다. 아래 핵심 코드에서는 필요한 시점에 다시 연결합니다.  

In [ ]:
driver.close()

## 교안 02 핵심 코드 이어서 보기

**의료 논문의 저장 결과 확인 -> 스키마와 근거 검사 -> 골드 비교 -> 청크 설정 변경 -> 결과 비교**를 실행합니다.  
본문에서 배운 방법을 약물의 치료 사용 관계인 `Compound -> TREATS -> Disease`에 적용합니다.  
교안 01의 `baseline_followalong.json`과 정답 목록 `followalong_gold.json`이 필요합니다. 새 커널에서는 이 절의 첫 코드부터 실행하세요.  
학생용에도 모든 코드를 채웠습니다. 새 추출은 4절에서 실행하며, 변경 결과는 `large_followalong.json`에 저장합니다.  

<img src="./images/paper_kg_validation.png" width="1000" alt="의료 논문의 저장 관계와 근거를 검사하고 같은 골드로 설정 변경 전후를 비교합니다">

### 공통 라이브러리 불러오기

이 핵심 코드에서 사용할 import를 먼저 실행합니다. 모델 생성과 DB 연결은 아래 해당 단계에서 실행합니다.

In [ ]:
# 필요한 라이브러리는 여기서 한 번 불러옵니다.
import json
from pathlib import Path
from pprint import pprint
import os
from urllib.parse import urlsplit
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from copy import deepcopy
from functools import partial
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from neo4j_graphrag.experimental.components.text_splitters.langchain import (
    LangChainTextSplitterAdapter,
)
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from uuid import uuid4

### 1. 평가할 실행과 저장 단계 확인

#### 1-1. 자료 경로 준비

의료 논문 저장본과 골드를 읽을 파일 경로를 준비합니다.  

In [ ]:
# 학생용은 현재 폴더, 정답은 한 단계 위 폴더의 자료를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(path):
    """JSON 파일 하나를 파이썬 사전 또는 목록으로 읽습니다."""
    # path는 파일 위치이며, UTF-8로 읽어 한글을 유지합니다.
    return json.loads(path.read_text(encoding="utf-8"))


def save_json(path, value):
    """실행 결과를 한글을 유지한 JSON 파일로 저장합니다."""
    # value는 저장할 사전이나 목록입니다. ensure_ascii=False는 한글을 문자 그대로 남깁니다.
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

#### 1-2. 의료 논문 실행 범위 확인

의료 논문의 원문, 설정과 저장 관계를 같은 실행에서 읽습니다.  

In [ ]:
# baseline_followalong.json: 교안 01에서 실제 실행한 설정, 원문, DB 조회 결과입니다.
follow_snapshot = read_json(output_dir / "baseline_followalong.json")
follow_doc = follow_snapshot["document"]
follow_rows = follow_snapshot["rows"]
print("평가 단계:", follow_snapshot["stage"])
print("원문:", follow_doc["doc_id"], "/ 저장 관계:", len(follow_rows))

### 2. 저장 관계의 스키마와 근거를 확인합니다

#### 2-1. 검사와 관계 키 함수 준비

스키마 검사, 원문 인용 검사와 관계 비교에 쓸 함수를 정의합니다.  

In [ ]:
def schema_ok(row, patterns):
    """한 행의 타입과 관계가 patterns에 지정한 조합인지 검사합니다."""
    return (row["subject_type"], row["relation"], row["object_type"]) in [
        tuple(pattern) for pattern in patterns
    ]


def evidence_ok(row, document):
    """근거가 비어 있지 않고 같은 문서의 전체 원문에 그대로 있으면 True입니다.

    document는 원문 문서 사전입니다. 개별 청크는 검사하지 않으므로,
    다른 청크의 문장을 인용했는지나 관계의 의미가 맞는지는 판정하지 않습니다.
    """
    # (1) row는 저장 관계 한 행, document는 대조할 출처 문서입니다.
    if row["source_doc_id"] != document["doc_id"]:
        return False

    # (2) 문자열이 아니거나 공백뿐이면 인용문으로 인정하지 않습니다.
    evidence = row["evidence"]
    if not isinstance(evidence, str) or not evidence.strip():
        return False

    # (3) 전체 문서에서 인용 문자열을 찾습니다. 출처 청크 내부의 일치는 별도 검사입니다.
    return evidence in document["text"]


def triple_key(row):
    """정밀도와 재현율에서 한 관계로 세는 (주어, 관계, 목적어)를 반환합니다."""
    return row["subject"], row["relation"], row["object"]

#### 2-2. 의료 논문 저장 관계의 두 비율

저장 관계 전체에서 두 검사의 통과 비율을 각각 계산합니다.  

In [ ]:
# 두 비율 모두 저장 관계 전체를 분모로 사용합니다. 검사 때문에 행을 삭제하지 않습니다.
# sum은 검사 결과 True를 1로 세어 통과 행 수를 구합니다.
follow_schema_count = sum(
    schema_ok(row, follow_snapshot["schema"]["patterns"]) for row in follow_rows
)
follow_evidence_count = sum(evidence_ok(row, follow_doc) for row in follow_rows)
follow_total = len(follow_rows)
print("스키마 준수 행:", follow_schema_count, "/", follow_total)
print("근거 원문 일치 행:", follow_evidence_count, "/", follow_total)
print(f"저장 관계의 스키마 준수율: {follow_schema_count / follow_total:.2%}")
print(f"저장 관계의 근거 원문 일치율: {follow_evidence_count / follow_total:.2%}")

#### 2-3. 의료 논문 관계의 근거 읽기

각 트리플의 근거와 연결된 원문 청크를 함께 읽습니다.  

In [ ]:
for row in follow_rows:
    print("트리플:", triple_key(row))
    print("근거:", row["evidence"])
    print("원문 일치:", evidence_ok(row, follow_doc))
    print("출처 청크:", row["chunk_texts"])
    print()

### 3. 같은 문서의 골드와 비교합니다

#### 3-1. 의료 논문 골드와 검토 범위 확인

원문 항목 4개에서 작성한 정답 트리플 3건을 읽습니다.  

In [ ]:
# followalong_gold.json: 의료 논문의 원문 항목 4개를 읽고 작성한 정답 트리플 3건입니다.
follow_gold = read_json(data_dir / "followalong_gold.json")
for row in follow_gold:
    print("원문 항목 ID:", row["paragraph_id"], "/ 정답 트리플:", triple_key(row))

# paragraphs는 정답을 만들 때 검토한 원문 항목 목록입니다.
# 원문 항목 하나에서 정답 트리플이 여러 건 나올 수 있으므로 두 수는 다를 수 있습니다.
print("검토한 원문 항목 수:", len(follow_doc["paragraphs"]))
print("정답으로 작성한 트리플 수:", len(follow_gold))

#### 3-2. 고유 관계를 골드와 비교하는 함수

고유 관계의 TP, FP, FN과 정밀도, 재현율, F1을 계산하는 함수를 정의합니다.  

In [ ]:
def score_relations(rows, gold):
    """같은 문서 범위의 고유 관계를 골드와 비교합니다.

    Args:
        rows: 평가할 저장 관계 전체. 근거 검사 실패 행도 포함합니다.
        gold: 해당 입력 문서의 고정 정답 목록.
    Returns:
        TP, FP, FN 집합과 정밀도, 재현율, F1을 담은 사전.
    """
    # (1) 같은 관계가 여러 행에 있어도 한 번만 세도록 집합을 만듭니다.
    # 검사 실패 행도 predicted에 포함하고, 골드 목록은 줄이지 않습니다.
    predicted = {triple_key(row) for row in rows}
    expected = {triple_key(row) for row in gold}
    # (2) 맞힘, 잘못 뽑음, 놓침을 집합 비교로 나눕니다.
    tp = predicted & expected  # 양쪽에 있음: 맞힌 관계
    fp = predicted - expected  # 추출에만 있음: 잘못 뽑은 관계
    fn = expected - predicted  # 골드에만 있음: 놓친 관계

    # (3) 정밀도는 추출 수, 재현율은 정답 수로 나눕니다. 분모가 없으면 None입니다.
    precision = len(tp) / len(predicted) if predicted else None
    recall = len(tp) / len(expected) if expected else None
    # 같은 TP, FP, FN으로 F1도 계산합니다.
    denominator = 2 * len(tp) + len(fp) + len(fn)
    f1 = 2 * len(tp) / denominator if denominator else None
    # (4) 점수와 오류 관계를 함께 반환해, 낮은 점수의 원인도 확인할 수 있게 합니다.
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

#### 3-3. 의료 논문의 맞힘과 누락 계산

의료 논문의 저장 관계 전체를 같은 원문에서 만든 골드와 비교합니다.  

In [ ]:
# 스키마나 근거 검사에서 실패한 저장 행도 비교합니다. 골드는 항상 같은 목록입니다.
follow_metrics = score_relations(follow_rows, follow_gold)
print(
    "TP / FP / FN:",
    len(follow_metrics["tp"]),
    "/",
    len(follow_metrics["fp"]),
    "/",
    len(follow_metrics["fn"]),
)
for metric in ["precision", "recall", "f1"]:
    value = follow_metrics[metric]
    print(metric + ":", "미산출" if value is None else f"{value:.4f}")
print("잘못 뽑은 관계:", sorted(follow_metrics["fp"]))
print("놓친 관계:", sorted(follow_metrics["fn"]))

### 4. 청크 크기와 겹침을 늘려 다시 검증합니다

#### 4-1. 재실행 연결과 모델 준비

실제 재추출에 사용할 Neo4j 연결과 모델을 준비합니다.  

In [ ]:
# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()


def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 목록으로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]


# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print(
    "Neo4j 연결 완료. 호스트:",
    connection_address.hostname,
    "/ 포트:",
    connection_address.port,
)


llm = OpenAILLM(
    model_name="gpt-5.6-luna",  # 관계를 추출할 모델 이름입니다.
)
embedder = OpenAIEmbeddings(
    model="text-embedding-3-large",  # 벡터를 만들 모델 이름입니다.
)
# partial은 이후 embed_query를 호출할 때 dimensions=768을 항상 함께 전달합니다.
embedder.embed_query = partial(embedder.embed_query, dimensions=768)

#### 4-2. 같은 추출 조건 복원

기준 실행의 스키마와 프롬프트를 그대로 읽습니다.  

In [ ]:
schema = follow_snapshot["schema"]
prompt_template = follow_snapshot["prompt_template"]
print(
    "기준 크기 / 겹침:",
    follow_snapshot["chunk_size"],
    "/",
    follow_snapshot["chunk_overlap"],
)
print("변경할 크기 / 겹침: 2000 / 400")

#### 4-3. 의료 논문의 넓어진 청크 확인

같은 의료 논문 원문을 청크 2000자, 겹침 400자로 나눕니다.  

In [ ]:
follow_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,  # 청크 하나의 최대 문자 수입니다.
    chunk_overlap=400,  # 청크 크기의 20%를 겹침 목표로 설정합니다.
)
# adapter는 LangChain 분할 결과를 빌더가 받는 청크 묶음으로 바꿉니다.
follow_splitter = LangChainTextSplitterAdapter(follow_text_splitter)
follow_chunks = await follow_splitter.run(text=follow_doc["text"])
for chunk in follow_chunks.chunks:
    print("청크:", chunk.index, "/ 문자 수:", len(chunk.text))
    print(chunk.text)
    print()
print("청크 수:", len(follow_chunks.chunks))

#### 4-4. 의료 논문을 다시 실행

새 실행 ID로 의료 논문를 다시 추출해 저장합니다.  

In [ ]:
follow_builder = SimpleKGPipeline(
    llm=llm,  # 원문에서 개체와 관계를 추출할 모델입니다.
    driver=driver,  # 생성한 그래프를 저장할 Neo4j 연결입니다.
    embedder=embedder,  # 청크의 검색용 벡터를 만드는 모델입니다.
    schema=deepcopy(schema),  # 허용 타입과 관계 규칙을 복사해 전달합니다.
    prompt_template=prompt_template,  # 추출 기준과 인용 규칙입니다.
    text_splitter=follow_splitter,  # 앞에서 확인한 청크 분할기입니다.
    from_file=False,  # 파일을 여는 대신 text로 받은 원문을 처리합니다.
    on_error="RAISE",  # 응답 처리에 실패하면 오류를 알리고 중단합니다.
    perform_entity_resolution=False,  # 같은 타입과 이름의 노드를 자동으로 합치지 않습니다.
)
print("준비한 청크 크기:", 2000)


# execution_id는 이번 실행의 결과만 조회하기 위한 번호이며 개체의 표준 ID가 아닙니다.
follow_execution_id = str(uuid4())
follow_result = await follow_builder.run_async(
    text=follow_doc["text"],  # 실제로 분할하고 추출할 원문입니다.
    file_path=follow_doc["url"],  # 문서 노드에 남길 출처 URL입니다. 접속하지 않습니다.
    # 원문 문서와 이번 실행을 구분할 정보를 문서 노드에 저장합니다.
    document_metadata={
        "source_doc_id": follow_doc["doc_id"],
        "execution_id": follow_execution_id,
    },
)
# ER을 끈 파이프라인의 마지막 단계는 writer입니다. 저장 실패를 완료로 처리하지 않습니다.
# writer는 그래프를 DB에 저장하는 구성요소이며 status는 저장 작업의 성공 여부입니다.
follow_writer_status = follow_result.result["writer"]["status"]
if follow_writer_status != "SUCCESS":
    raise RuntimeError(follow_result.result["writer"])
print("완료한 실행 ID:", follow_execution_id)
print("DB 저장 상태:", follow_writer_status)

#### 4-5. 조회 함수 준비

지정한 실행의 관계와 청크를 읽는 함수를 정의합니다.  

In [ ]:
def read_relations(execution_id):
    """지정한 실행에서 저장한 개체 간 관계를 평가용 사전 목록으로 읽습니다.

    Args:
        execution_id (str): 빌더를 실행할 때 Document에 저장한 실행 ID.
            follow_execution_id처럼, 조회하려는 실행에서 사용한 값을 전달합니다.

    Returns:
        list[dict]: 관계 ID별 트리플과 근거, 출처 정보. 결과가 없으면 [].
            같은 관계의 청크 원문은 chunk_texts 목록에 모읍니다.

    """
    return run_cypher(
        """
    // (1) 실행 ID로 문서 범위를 고르고, 그 문서의 청크와 주어 개체를 찾습니다.
    MATCH (d:Document {execution_id: $execution_id})<-[:FROM_DOCUMENT]-(c:Chunk)<-[:FROM_CHUNK]-(s:__Entity__)
    // (2) 같은 청크에 연결된 목적어를 찾습니다. 관계 타입은 제한하지 않습니다.
    MATCH (s)-[r]->(o:__Entity__)-[:FROM_CHUNK]->(c)
    // (3) AS 오른쪽 이름이 반환 사전의 키가 됩니다.
    RETURN
        elementId(r) AS relationship_id, // DB 안에서 관계를 구분하는 ID입니다.
        s.name AS subject, // 주어 노드의 이름입니다.
        head([x IN labels(s) WHERE NOT x STARTS WITH '__']) AS subject_type, // 관리 레이블을 제외한 첫 타입입니다.
        type(r) AS relation, // 주어에서 목적어로 향하는 관계 타입입니다.
        o.name AS object, // 목적어 노드의 이름입니다.
        head([x IN labels(o) WHERE NOT x STARTS WITH '__']) AS object_type, // 관리 레이블을 제외한 첫 타입입니다.
        coalesce(r.evidence, '') AS evidence, // 근거 인용문이며, 없으면 빈 문자열입니다.
        d.source_doc_id AS source_doc_id, // 원본 문서 ID입니다. 실행 ID와 다릅니다.
        collect(DISTINCT c.text) AS chunk_texts // 연결된 청크 원문을 중복 없이 모읍니다.
    ORDER BY subject, relation, object, relationship_id
    """,
        execution_id=execution_id,
    )


def read_chunks(execution_id):
    """지정한 실행에서 저장한 청크의 순서, 원문, 임베딩 차원 수를 읽습니다.

    Args:
        execution_id (str): 빌더를 실행할 때 Document에 저장한 실행 ID.
            문서 이름이나 source_doc_id가 아니라 follow_execution_id 같은 실행 값을 씁니다.

    Returns:
        list[dict]: 청크별 원문과 임베딩 차원 정보. 순번순으로 정렬하며, 없으면 [].

    """
    return run_cypher(
        """
    // (1) 지정한 실행의 문서에 연결된 청크만 고릅니다.
    MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document {execution_id: $execution_id})
    // (2) 청크 하나를 사전 하나로 읽습니다. AS 오른쪽이 사전의 키입니다.
    RETURN
        c.index AS index, // 문서 안의 청크 순번입니다. 0부터 시작합니다.
        c.text AS text, // 청크에 저장된 원문입니다.
        size(c.embedding) AS dimensions // 벡터 원소 수, 즉 임베딩 차원입니다.
    // 원문을 읽는 순서대로 확인할 수 있게 청크 순번으로 정렬합니다.
    ORDER BY index
    """,
        execution_id=execution_id,
    )

#### 4-6. 의료 논문 변경 결과 저장

기준 실행은 follow_before로 유지하고 변경 결과를 별도 파일에 저장합니다.  

In [ ]:
follow_before = follow_snapshot
# 실행 ID가 같은 DB 관계와 청크를 각각 읽습니다. 아직 노드를 통합하기 전입니다.
follow_rows = read_relations(follow_execution_id)
follow_stored_chunks = read_chunks(follow_execution_id)
print("저장 관계 행 수:", len(follow_rows), "/ 청크 수:", len(follow_stored_chunks))
for chunk in follow_stored_chunks:
    print("청크:", chunk["index"], "/ 임베딩 차원:", chunk["dimensions"])
for row in follow_rows:
    print("관계:", row["subject"], "->", row["relation"], "->", row["object"])
    print("근거:", row["evidence"])
    print()

# 문서, 설정과 조회 결과를 함께 저장해 교안 02에서 같은 실행을 평가합니다.
follow_snapshot = {
    # 평가 대상이 중복 노드 통합 전 결과임을 기록합니다.
    "stage": "허용하지 않은 노드·관계·속성을 가지치기한 뒤 DB에 저장한 결과(중복 노드 통합 전)",
    # 어느 실행에서 만든 결과인지 구분합니다.
    "execution_id": follow_execution_id,
    # 전체 원문: 근거 인용을 검사하고, 같은 문서로 다시 추출할 때 사용합니다.
    "document": follow_doc,
    # 실행 설정: 비교 실험에서 바꿀 조건과 유지할 조건을 확인합니다.
    "chunk_size": 2000,
    "chunk_overlap": 400,
    "text_splitter": "RecursiveCharacterTextSplitter",
    "model": "gpt-5.6-luna",
    "embedding_model": "text-embedding-3-large",
    "dimensions": 768,
    "schema": schema,
    "prompt_template": prompt_template,
    "perform_entity_resolution": False,
    # 추출 관계: 스키마와 근거를 검사하고 골드와 비교합니다.
    "rows": follow_rows,
    # 당시 청크: 원문이 나뉜 위치와 변경 전후의 청크 수와 내용을 확인합니다.
    "chunks": follow_stored_chunks,
}
save_json(output_dir / "large_followalong.json", follow_snapshot)
print("저장 파일:", output_dir / "large_followalong.json")

#### 4-7. 의료 논문 설정 전후 비교

두 의료 논문 실행을 같은 원문과 골드로 평가합니다.  

In [ ]:
for label, snapshot in [("500자", follow_before), ("2000자", follow_snapshot)]:
    rows = snapshot["rows"]
    document = snapshot["document"]
    print(
        "청크 설정:", label, "/ 저장 행:", len(rows), "/ 청크:", len(snapshot["chunks"])
    )
    print("크기 / 겹침:", snapshot["chunk_size"], "/", snapshot["chunk_overlap"])
    schema_count = sum(schema_ok(row, snapshot["schema"]["patterns"]) for row in rows)
    evidence_count = sum(evidence_ok(row, document) for row in rows)
    print("스키마 준수:", schema_count, "/", len(rows))
    print("근거 원문 일치:", evidence_count, "/", len(rows))
    if rows:
        print(
            f"스키마 준수율: {schema_count / len(rows):.2%} / 근거 원문 일치율: {evidence_count / len(rows):.2%}"
        )
    else:
        print("스키마 준수율: 미산출 / 근거 원문 일치율: 미산출")
    metrics = score_relations(rows, follow_gold)
    print(
        "TP / FP / FN:",
        len(metrics["tp"]),
        "/",
        len(metrics["fp"]),
        "/",
        len(metrics["fn"]),
    )
    for metric in ["precision", "recall", "f1"]:
        value = metrics[metric]
        print(metric + ":", "미산출" if value is None else f"{value:.4f}")
    print("FP:", sorted(metrics["fp"]))
    print("FN:", sorted(metrics["fn"]))
    print()

#### 4-8. 본문의 연결 종료

의료 논문의 비교가 끝나면 DB 연결을 닫습니다.  

In [ ]:
driver.close()

## 부록. LLMGraphTransformer와 입력 및 반환값을 비교합니다

<img src="./images/pandas_builder_vs_transformer.png" width="1000" alt="SimpleKGPipeline은 분할, 임베딩, 추출과 가지치기 후 Neo4j에 저장하고 실행 결과 정보를 반환합니다. LLMGraphTransformer는 원문 Document에서 추출해 GraphDocument 목록을 반환하며 DB 저장은 별도입니다.">

**`GraphDocument`는 DB가 아닌 Python의 추출 결과 객체입니다.** `.nodes`는 노드, `.relationships`는 관계, `.source`는 입력 문서입니다.  
`LLMGraphTransformer`에 청크 분할, 임베딩, DB 저장과 노드 통합이 필요하면 별도로 연결합니다.  

### 허용한 타입과 관계만 결과에 남깁니다

- **`strict_mode=True`(기본값):** 허용 목록 밖 노드와 관계를 반환 전에 제외합니다.
- **아래 설정:** `Release -> FIXES_API -> ApiElement`는 남고, `MENTIONS` 관계는 제외됩니다.
- **반환 목록:** 이미 검사를 거친 결과입니다. 남은 관계의 의미가 맞는지는 원문과 대조합니다.
[공식 API](https://reference.langchain.com/python/langchain-neo4j/graph_transformers/llm/LLMGraphTransformer)  

#### 같은 작은 원문으로 반환 관계 확인

- 입력 `Document`는 `page_content`에 원문, `metadata`에 출처 정보를 담는 LangChain 객체입니다. DB의 문서 노드를 만드는 코드는 아닙니다.
- `aconvert_to_graph_documents`에 문서 목록을 넣고 `await`로 결과를 받습니다.
- `graph_documents[0]`은 첫 문서의 결과입니다. `.relationships`에서 관계의 양 끝과 근거를 읽습니다.

In [ ]:
# 교안 01과 같은 입력과 허용 관계를 사용하되 반환 자료의 구조를 살펴봅니다.
transformer = LLMGraphTransformer(
    llm=ChatOpenAI(model="gpt-5.6-luna"),  # 추출을 수행할 채팅 모델입니다.
    allowed_nodes=["Release", "ApiElement"],  # 허용할 노드 타입입니다.
    allowed_relationships=[
        ("Release", "FIXES_API", "ApiElement")
    ],  # 관계의 방향과 양 끝 타입입니다.
    node_properties=["name"],  # 노드에 기록할 속성입니다.
    relationship_properties=["evidence"],  # 관계에 기록할 원문 근거입니다.
    strict_mode=True,  # 허용한 노드 타입과 관계 조합에 맞는 결과만 남깁니다.
    # 기본 추출 지시에 추가할 포함 기준과 인용 규칙입니다.
    additional_instructions="원문에서 해당 버전이 수정했다고 명시한 API만 추출하세요. API 이름과 근거를 원문 그대로 유지하고 설치 옵션 추가는 제외하세요.",
)
graph_documents = await transformer.aconvert_to_graph_documents(
    [
        # page_content는 원문, metadata는 원본 Document에 보관할 출처 정보입니다.
        Document(
            page_content=demo_doc["text"],
            metadata={"source_doc_id": demo_doc["doc_id"]},
        )
    ]
)
for relation in graph_documents[0].relationships:
    print("관계:", relation.source.id, "->", relation.type, "->", relation.target.id)
    print("근거:", relation.properties.get("evidence", ""))
    print()